# WordPiece from scratch — Exercise

Companion to [Tokenization § WordPiece](https://tanulsingh.github.io/blog/tokenization#wordpiece).

WordPiece is BPE's cousin — used by BERT, DistilBERT, ELECTRA. The algorithm is **almost** identical, with two key differences:

1. **Merge criterion**: WordPiece merges the pair with highest **PMI** (pointwise mutual information), not raw frequency.
2. **Encoding**: WordPiece uses **greedy longest-match-first** lookup, not replay of merges. Continuation pieces get a `##` prefix.

You'll build it by modifying your BPE code from notebook 01.

**Pre-requisite:** complete `01_bpe_exercise.ipynb` first.

## Setup

In [ ]:
# TODO: imports + bring in pre_tokenize, merge_pair from notebook 01

## TODO 1 — PMI-based pair scoring

For each candidate pair `(a, b)`, the PMI score is:

$$\text{score}(a, b) = \frac{\text{count}(ab)}{\text{count}(a) \cdot \text{count}(b)}$$

This measures "how much MORE often does this pair co-occur than we'd expect if `a` and `b` were independent?"

**Concrete example.** If `'t'` appears 10,000 times and `'h'` appears 8,000 times in a corpus of 50,000 token positions, the *expected* co-occurrence under independence is roughly `10000 × 8000 / 50000 = 1600`. If the actual `(t, h)` count is 3,000, the score is `3000 / (10000 × 8000) ≈ very small` — but it's the relative ranking across pairs that matters.

Meanwhile a rare pair like `(q, u)` — where `q` appears only 200 times but is followed by `u` 195 times — gets a much higher PMI score. WordPiece prefers `(q, u)` over `(t, h)` because their association is much stronger.

**Implementation hint:** along with `get_pair_counts`, you'll also need `get_token_counts` — how often each individual token appears across the corpus.

In [ ]:
def get_token_counts(word_freqs: dict[tuple[str, ...], int]) -> dict[str, int]:
    """Count individual tokens (not pairs) across the corpus."""
    # TODO 1a: implement — for each (word, freq), accumulate freq into each token's count
    pass


def get_pmi_scores(
    word_freqs: dict[tuple[str, ...], int],
) -> dict[tuple[str, str], float]:
    """Score each adjacent pair by PMI = count(ab) / (count(a) * count(b))."""
    # TODO 1b: get pair_counts (reuse get_pair_counts from notebook 01)
    # TODO 1c: get token_counts via get_token_counts above
    # TODO 1d: for each pair (a, b), compute pair_counts[(a,b)] / (token_counts[a] * token_counts[b])
    pass

In [ ]:
# Sanity check
corpus = ["the cat sat", "the bat sat", "queen quick"]
word_freqs = pre_tokenize(corpus)
scores = get_pmi_scores(word_freqs)
for pair, s in sorted(scores.items(), key=lambda x: -x[1])[:5]:
    print(pair, '→', round(s, 5))
# Expected: pairs like ('q', 'u') should score very high (q always followed by u in 'queen','quick')
#           pairs like ('t', 'h') will score lower despite higher raw counts

## TODO 2 — `WordPieceTokenizer.train`

Nearly identical to BPE's training loop, with two changes:

1. Pick the pair with maximum **PMI score**, not raw count.
2. (Optional) Apply WordPiece's **`##` convention** when adding tokens to the vocabulary: tokens that aren't word-starts get a `##` prefix. This is more about the *output format* than the algorithm itself; you can skip it for the from-scratch version and just track tokens as plain strings.

In [ ]:
class WordPieceTokenizer:
    """WordPiece tokenizer — BPE variant with PMI-based merging."""

    def __init__(self):
        self.vocab: set[str] = set()
        # Note: WordPiece doesn't keep ordered merges the same way BPE does, because encoding
        # uses greedy longest-match (TODO 3) rather than replay. The vocabulary is the only output.

    def train(self, corpus: list[str], vocab_size: int):
        # TODO 2a: word_freqs = pre_tokenize(corpus)
        # TODO 2b: initialize self.vocab with all unique characters
        # TODO 2c: while len(self.vocab) < vocab_size:
        #             scores = get_pmi_scores(word_freqs)
        #             if not scores: break
        #             best_pair = max(scores, key=scores.get)
        #             word_freqs = merge_pair(best_pair, word_freqs)
        #             self.vocab.add(best_pair[0] + best_pair[1])
        pass

    def encode(self, text: str) -> list[str]:
        # implemented in TODO 3
        raise NotImplementedError("TODO 3")

## TODO 3 — Greedy longest-match encoding

Unlike BPE, WordPiece doesn't replay merges. Instead, at inference, it does **greedy longest-match-first**:

```
For each word:
    Try the entire word — is it in the vocab? If yes, output it. Done.
    Otherwise, try the word minus its last character. In vocab?
    Keep shrinking from the end until you find the longest prefix in the vocab.
    Output that piece. Recurse on the remainder (with `##` prefix to mark continuation).
    If nothing matches even at length 1, output the unknown token (e.g. `[UNK]`).
```

**Example trace** for `"unhappiness"`:
1. Try `"unhappiness"` — not in vocab
2. Try `"unhappines"`, `"unhappine"`, ... → eventually find `"un"` is in vocab
3. Output `"un"`. Now process remainder `"happiness"`
4. Try `"happiness"` — say it's in vocab (since it's a common stem)
5. Output `"##happiness"` (with `##` prefix because it's a continuation)

Result: `["un", "##happiness"]`

In [ ]:
# Add this method to your WordPieceTokenizer class above:
def encode(self, text):
    # TODO 3: implement greedy longest-match-first
    # For each word in pre_tokenize(text):
    #   - Loop: try longest prefix in vocab; if first piece in word, no '##' prefix;
    #     otherwise prepend '##'
    #   - If no prefix matches, emit '[UNK]' for the whole word
    pass

WordPieceTokenizer.encode = encode   # patch onto the class

In [ ]:
# Sanity check
tok = WordPieceTokenizer()
tok.train(["the cat sat", "the dog ran", "the cat sat"] * 10, vocab_size=20)
print('vocab:', sorted(tok.vocab))
print('encode("the cat"):', tok.encode("the cat"))
print('encode("thecatxyz"):', tok.encode("thecatxyz"))   # made-up word, should fragment

## Run the tests

In [ ]:
from tests import run_wordpiece_tests
# run_wordpiece_tests(WordPieceTokenizer, get_pmi_scores)